In [3]:
import sys

base_dir = 'RAVE'
sys.path.append(f'{base_dir}')

from absl import logging
import pdb
import torch, torchaudio, argparse, os, tqdm, re, gin
import cached_conv as cc
import IPython.display as ipd

import rave

torch.set_float32_matmul_precision('high')
cc.use_cached_conv(False)

print(torch.__version__)
print(torchaudio.__version__)

2.12.0
2.11.0


In [4]:
device = "cpu"

if torch.cuda.is_available():
    device = "cuda"

elif torch.backends.mps.is_available():
    device = "mps"

print(f'Using device: {device}')

Using device: mps


## Loading a checkpoint and running inference

Load a trained RAVE run, then run a forward pass (encode → sample latent → decode) to reconstruct an audio signal.

In [5]:
# Path to a trained run folder (or a .ckpt file directly)
run_path = "aw_guitarDm_v3"

# Locate and parse the gin config that defines the model architecture
config_file = rave.core.search_for_config(run_path)
assert config_file is not None, f"No config.gin found in {run_path}"
gin.parse_config_file(config_file)

# Locate the latest checkpoint inside the run folder
ckpt_path = rave.core.search_for_run(run_path)
print(f"Config:     {config_file}")
print(f"Checkpoint: {ckpt_path}")

# Instantiate the model from the gin config and load the trained weights
model = rave.RAVE()
checkpoint = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(checkpoint["state_dict"], strict=False)
model = model.eval().to(device)

print(f"Sample rate:  {model.sr}")
print(f"Channels:     {model.n_channels}")
print(f"Latent size:  {model.latent_size}")

Config:     /Users/jasperrr/PhD/26-06-rave-mlx/RAVE/aw_guitarDm_v3/config.gin
Checkpoint: aw_guitarDm_v3/version_6/checkpoints/epoch-epoch=38591.ckpt


/Users/jasperrr/PhD/26-06-rave-mlx/RAVE/.venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/Users/jasperrr/PhD/26-06-rave-mlx/RAVE/.venv/lib/python3.11/site-packages/torchaudio/transforms/_transforms.py:95: UserWarning: `return_complex` argument is now deprecated and is not effective.`torchaudio.transforms.Spectrogram(power=None)` always returns a tensor with complex dtype. Please remove the argument in the function call.
  warnings.warn(


Sample rate:  44100
Channels:     1
Latent size:  128


In [6]:
# Provide an audio file to reconstruct, or leave as None to use random noise
audio_path = "aw_guitarDm_v3/em-guitar.wav"  # e.g. "path/to/audio.wav"

if audio_path is not None:
    # torchaudio.load now routes through TorchCodec; soundfile is a lighter,
    # dependency-free way to read a WAV into a (channels, time) tensor.
    import soundfile as sf
    wav, sr = sf.read(audio_path, dtype="float32", always_2d=True)  # (time, channels)
    x = torch.from_numpy(wav.T)  # -> (channels, time)
    if sr != model.sr:
        x = torchaudio.functional.resample(x, sr, model.sr)
    # keep the expected number of channels and add a batch dim -> (B, C, T)
    x = x[:model.n_channels].unsqueeze(0)
else:
    # 2 seconds of low-level noise as a stand-in input
    x = torch.randn(1, model.n_channels, 2 * model.sr) * 0.1

x = x.to(device)
print(f"Input shape: {tuple(x.shape)}")

Input shape: (1, 1, 352800)


In [7]:
# Forward pass: encode -> sample the latent -> decode (full reconstruction)
with torch.no_grad():
    y = model(x)

print(f"Output shape: {tuple(y.shape)}")

# The latent representation can also be obtained on its own
with torch.no_grad():
    z = model.encode(x)
    z = model.encoder.reparametrize(z)[0]

print(f"Latent shape: {tuple(z.shape)}")

Output shape: (1, 1, 354304)
Latent shape: (1, 128, 173)


In [8]:
# Listen to the reconstruction (first channel)
y_cpu = y.squeeze(0).cpu()
ipd.display(ipd.Audio(y_cpu[0].numpy(), rate=int(model.sr)))